# Saving module configurations & saving a complete pipeline

### Saving interactively added module configurations to a BioPAP config file

In [28]:
%pip install -e ..

Obtaining file:///home/maarten/Documents/Projects/BioPAP/BioPAP
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for BioPAP (pyproject.toml) ... done
  Created wheel for BioPAP: filename=biopap-0.0.1-0.editable-py3-none-any.whl size=3008 sha256=b740248cc71d3136afde8a170581b4813a852f9de3b2af6b1c1a9a41d0ee5710
  Stored in directory: /tmp/pip-ephem-wheel-cache-2s3frpmv/wheels/a9/55/80/ba98fe05e3d05116737c29a1c667eb5bffec478c135f508c7b
Successfully built BioPAP
  Attempting uninstall: BioPAP
    Found existing installation: BioPAP 0.0.1
    Uninstalling BioPAP-0.0.1:
      Successfully uninstalled BioPAP-0.0.1
Note: you may need to restart the kernel to use updated packages.


In [31]:
from biopap.modules.anonymize import Anonymize
from biopap.modules.apply_ica import ApplyICA
from biopap.modules.artifact_reject import ArtifactReject
from biopap.modules.baseline_correction import BaselineCorrection
from biopap.modules.bids import BIDSConversion
from biopap.modules.blink_detection import BlinkDetection
from biopap.modules.downsample import Downsample
from biopap.modules.eog_channels import EOGChannels
from biopap.modules.iir_filter import IIRFilter
from biopap.modules.load_data import LoadData
from biopap.modules.plot import Plot, PlotTypes
from biopap.modules.rereference import Rereference
from biopap.modules.segmentation import Segmentation
from biopap.modules.save_output import SaveOutput
from biopap.modules.set_channel_type_and_montage import SetChannelTypeAndMontage

from biopap.pipeline import Pipeline
from biopap.modules.helpers import save_modules_config

In [21]:
# Have a data file and load it
data_file = '/my/data/ABC1234.bdf'

load = LoadData(file_path=data_file)

# Add the modules you want to use, adding and testing the configuration for
# each until you are satisfied that each module's settings are correct.

ano = Anonymize(
    raw=load.raw,
    daysback=100,
    experimenter='J. Jansen',
    participant_id='AB00123')

Extracting EDF parameters from /data/BioPAP/data/CBN1BD071212-Cropped.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 3276799  =      0.000 ...  1600.000 secs...


In [22]:
# You can use both the standard variables - in this case `ref_channels` -, as well as additional
# parameters to be passed on the the underlying mne-function, like `ch_type` in the example below.

ref = Rereference(
    raw=ano.processed,
    ref_channels=['EXG1', 'EXG2'],
    ch_type='auto')

EEG channel type selected for re-referencing
Applying a custom ('EEG',) reference.


In [23]:
plt = Plot(
    raw=ano.processed,
    save_folder='./',
    save_postfix='IRR')

eog = EOGChannels(
    raw=ref.processed,
    ref_channels=['hEOG', 'vEOG'],
    anode_channels=['EXG3','EXG5'],
    cathode_channels=['EXG4','EXG6'])

channel_types = {
    'misc': ['EXG1','EXG2','EXG3','EXG4','EXG5','EXG6','EXG7','EXG8','Erg1','Erg2','Plet'],
    'gsr': ['GSR1', 'GSR2'],
    'resp': ['Resp'],
    'temperature': ['Temp']
}

montage = SetChannelTypeAndMontage(
    raw=eog.processed,
    channel_types=channel_types,
    mne_standard_montage='biosemi64'
)

flt = IIRFilter(
    raw=montage.processed,
    l_freq=0.1,
    h_freq=35)

dwn = Downsample(
    raw=flt.processed,
    sfreq=500)  # Downsample to 500 Hz

blink = BlinkDetection(raw=dwn.processed)

Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
2042 events found on stim channel Status
Event IDs: [  1   2   3   4   5   6   9  10  11  12  13  14  15  16  17  18  19  20
  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37  38
  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54  55  56
  57  58  59  60  61  62  63  64  65  66  67  68  69  96  97  98  99 101
 102 103 104 105 106 121 122 123 124 125 126 141 142 143 144 145 146 161
 162 163 164 165 201 203 211 212 213 231 232 233 240 241 253]
wrote plot to 'AB00123_plot_IRR.png'
EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=2, n_times=3276800
    Range : 0 ... 3276799 =      0.000 ...  1600.000 secs
Ready.
Added the following bipolar channels:
hEOG, vEOG


2026-06-11 11:39:12,711:WARNING:warnings:set_channel_type_and_montage:The unit for channel(s) EXG1, EXG2, EXG3, EXG4, EXG5, EXG6, EXG7, EXG8, Erg1, Erg2, Plet has changed from V to NA.
2026-06-11 11:39:12,711:WARNING:warnings:set_channel_type_and_montage:The unit for channel(s) GSR1, GSR2 has changed from V to S.
2026-06-11 11:39:12,711:WARNING:warnings:set_channel_type_and_montage:The unit for channel(s) Temp has changed from V to C.


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.1 - 35 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 0.10, 35.00 Hz: -6.02, -6.02 dB

Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
2042 events found on stim channel Status
Event IDs: [  1   2   3   4   5   6   9  10  11  12  13  14  15  16  17  18  19  20
  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37  38
  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54  55  56
  57  58  59  60  61  62  63  64  65  66  67  68  69  96  97  98  99 101
 102 103 104 105 106 121 122 123 124 125 126 141 142 143 144 145 146 161
 162 163 164 165 201 203 211 212 213 231 232 233 240 241 253]
Finding

2026-06-11 11:39:26,257:WARNING:warnings:downsample:Resampling of the stim channels caused event information to become unreliable. Consider finding events on the original data and passing the event matrix as a parameter.


Using EOG channels: hEOG, vEOG
EOG channel index for this subject is: [80 81]
Filtering the data to remove DC offset to help distinguish blinks from saccades
Selecting channel vEOG for blink detection
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 10.25 Hz)
- Filter length: 5000 samples (10.000 s)

Now detecting blinks and generating corresponding events
Found 235 significant peaks
Number of EOG events detected: 235


In [24]:
# Make sure when you have multiple instances of the same module
# in your pipeline to give each instance a different name (`plt` vs `plt2`).

plt2 = Plot(
    raw=blink.processed,
    title="Show blink annotations")

Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
2041 events found on stim channel Status
Event IDs: [  1   2   3   4   5   6   9  10  11  12  13  14  15  16  17  18  19  20
  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37  38
  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54  55  56
  57  58  59  60  61  62  63  64  65  66  67  68  69  96  97  98  99 101
 102 103 104 105 106 121 122 123 124 125 126 141 142 143 144 145 146 161
 162 163 164 165 201 203 211 212 213 231 232 233 240 241 253]
Channels marked as bad:
none


In [25]:
seg = Segmentation(
    raw=blink.processed,  
    stim_channel='Status',  
    start=-0.2,  
    end=0.8,     
    baseline=(None, 0),
    reference_markers={'range': '1-69'})

artifact = ArtifactReject(
    raw=seg.processed,
    mode='interpolation',
    max_gradient_uv=50,
    max_amplitude_uv=75,
    min_activity_uv=0.5,
    interval_ms=100)

baseline = BaselineCorrection(
    raw=artifact.processed,
    baseline=(-0.2, 0))

pre_ica_save = SaveOutput(
    raw=baseline.processed,
    export_formats='fif',
    output_file='./partially_processed.fif',
    overwrite=True,
    intermediate='_before_ICA')

ica_result = ApplyICA(
    raw=baseline.processed,
    n_components=None, # use all components
    method='fastica',
    silent=False)

plt3 = Plot(
    raw=ica_result.processed,
    title="Preprocessed data")

Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
2041 events found on stim channel Status
Event IDs: [  1   2   3   4   5   6   9  10  11  12  13  14  15  16  17  18  19  20
  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37  38
  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54  55  56
  57  58  59  60  61  62  63  64  65  66  67  68  69  96  97  98  99 101
 102 103 104 105 106 121 122 123 124 125 126 141 142 143 144 145 146 161
 162 163 164 165 201 203 211 212 213 231 232 233 240 241 253]
Not setting metadata
540 matching events found
Setting baseline interval to [-0.2, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 540 events and 501 original time points ...
20 bad epochs dropped
Fp1: bad in 3 epochs
AF7: bad in 2 epochs
AF3: bad in 1 epoch

2026-06-11 11:39:35,313:WARNING:warnings:save_output:This filename (partially_processed__before_ICA.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
2026-06-11 11:39:35,494:WARNING:warnings:apply_ica:The epochs you passed to ICA.fit() were baseline-corrected. However, we suggest to fit ICA only on data that has been high-pass filtered, but NOT baseline-corrected.


wrote 'partially_processed__before_ICA.fif' (85,479,520)
Fitting ICA to data using 64 channels (please be patient, this may take a while)
Selecting by non-zero PCA components: 64 components
Fitting ICA took 24.8s.


2026-06-11 11:40:00,313:WARNING:warnings:apply_ica:The data you passed to ICA.apply() was baseline-corrected. Please note that ICA can introduce DC shifts, therefore you may wish to consider baseline-correcting the cleaned data again.


Applying ICA to Epochs instance
    Transforming to ICA space (64 components)
    Zeroing out 0 ICA components
    Projecting back using 64 PCA components
You seem to have overlapping epochs. Some event lines may be duplicated in the plot.


2026-06-11 11:40:00,601:WARNING:warnings:plot:More events than default colors available. You should pass a list of unique colors.


Dropped 0 epochs: 
The following epochs were marked as bad and are dropped:
[]
Channels marked as bad:
none


In [26]:
save_output = SaveOutput(
    raw=ica_result.processed,
    export_formats='fif',
    output_file='./processed_data-epo.fif',
    intermediate=False,
    overwrite=True)

wrote 'processed_data-epo.fif' (85,479,520)


Once you are done with adding modules and editing their configuration, the settings for the complete pipeline can be saved to a configuration file by calling the `save_modules_config` helper function.

The order of the modules is important, as this will also be the order in which the modules will be added when a new pipeline is created from the saved configuration file.

Note that you only need to pass the function a list of the modules that you actually want added to the config. For instance, the example below omits `plt2`, the second plotting step.

Also note that it is not necessary to save the settings from the `Load` module, as that module is also integrated in the `Pipeline` class.

In [27]:
cfg_file = './config.yaml'
save_modules_config(
    modules=[ano, ref, plt, eog, flt, dwn, seg, artifact, baseline, pre_ica_save, ica_result, plt3, save_output], 
    config_path=cfg_file
)

By loading a saved config file into a new `Pipeline`-instance, you automatically create the complete pipeline with all the specified settings.

In [17]:
cfg_file = './config.yaml'
data_file = '/data/BioPAP/data/CBN1BD071212-Cropped.bdf'

pipe = Pipeline(
    config_file=cfg_file,
    input_file=data_file)

pipe.to_file(
    save_path=args.stored_pipeline,
    label=args.label,
    author=args.author,
    overwrite=True
)

FileNotFoundError: [Errno 2] No such file or directory: './config.yaml'

In [15]:
dir(pipe)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_config',
 '_config_file',
 '_initial_data',
 '_processed_data',
 '_steps',
 '_subject_id',
 'add_step',
 'check_steps',
 'config',
 'from_file',
 'initial_data',
 'inspect',
 'load_data',
 'processed_data',
 'read_config',
 'read_saved',
 'reset',
 'run',
 'steps',
 'subject_id',
 'to_file',
 'verbose']

In [16]:
pipe.inspect()

TypeError: Pipeline.inspect() missing 1 required positional argument: 'save_path'

### Saving the complete pipeline to file

In [ ]:
|